# 10 · Discontinuous Galerkin & HDG

Continuous ($H^1$) elements struggle when **convection dominates** — the solution oscillates.
**Discontinuous Galerkin (DG)** breaks the continuity: each element carries its own polynomial,
and elements communicate through **numerical fluxes** (here, **upwind**). **Hybrid DG (HDG)**
adds an unknown on the **facets** and **statically condenses** the element interiors away,
shrinking the global system — the very method behind Part III's plume and fan.

In [ ]:
# --- Google Colab: install NGSolve on first run (a no-op anywhere else) -------
import sys
if "google.colab" in sys.modules:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "--pre",
                    "ngsolve", "webgui_jupyter_widgets"], check=True)

In [ ]:
from ngsolve import *
from netgen.geom2d import SplineGeometry
from ngsolve.webgui import Draw
import numpy as np
import matplotlib.pyplot as plt

## 1. A convection-dominated problem — where $H^1$ wobbles

Same double-glazing wind as unit 8, but now with **tiny diffusion** $\varepsilon=10^{-3}$:
$-\varepsilon\Delta u + \mathbf b\!\cdot\!\nabla u = 0$, hot right wall ($u=1$), cold left
($u=0$). The wind is **tangential** to the boundary ($\mathbf b\!\cdot\!\mathbf n=0$). A
standard continuous `H1` discretisation **overshoots wildly** — values far outside $[0,1]$.

In [ ]:
geo = SplineGeometry()
geo.AddRectangle((-1, -1), (1, 1), bcs=["bottom", "right", "top", "left"])
mesh = Mesh(geo.GenerateMesh(maxh=0.08))
eps, order = 1e-3, 3
wind = CF((2*y*(1-x*x), -2*x*(1-y*y)))
ud = IfPos(x, 1.0, 0.0)                                 # right wall (x>0) hot=1, left wall cold=0
n = specialcf.normal(2); h = specialcf.mesh_size; alpha = 4*order*order

V1 = H1(mesh, order=order, dirichlet="right|left")
u, v = V1.TnT()
a1 = BilinearForm(eps*grad(u)*grad(v)*dx + (wind*grad(u))*v*dx).Assemble()
uH1 = GridFunction(V1); uH1.Set(ud, definedon=mesh.Boundaries("right|left"))
uH1.vec.data += a1.mat.Inverse(V1.FreeDofs()) * (-a1.mat*uH1.vec).Evaluate()
print(f"H1:  u in [{min(uH1.vec):.2f}, {max(uH1.vec):.2f}]   <- should be [0,1]!")

## 2. Upwind DG — broken elements, fluxes across edges

We use a **broken** space `L2(..., dgjumps=True)` (no continuity). Two ingredients on the
**interior edges** (`dx(skeleton=True)`):

* **convection** by an **upwind** flux — the edge value is taken from whichever side the wind
  blows *from*, `IfPos(b·n, u, u.Other())`;
* **diffusion** by the **symmetric interior penalty** (consistency + symmetry + an
  $\alpha/h$ penalty on the jump).

Dirichlet walls are imposed **weakly** (Nitsche). The result stays essentially within $[0,1]$.

In [ ]:
V2 = L2(mesh, order=order, dgjumps=True)
u, v = V2.TnT()
jmp = lambda w: w - w.Other()
mn = lambda gw: 0.5*(gw + gw.Other())*n
dS = dx(skeleton=True)                                  # interior edges
dD = ds(skeleton=True, definedon=mesh.Boundaries("right|left"))   # Dirichlet edges
bn = wind*n
a2 = BilinearForm(V2)
a2 += eps*grad(u)*grad(v)*dx                                                       # diffusion (volume)
a2 += eps*(-mn(grad(u))*jmp(v) - mn(grad(v))*jmp(u) + alpha/h*jmp(u)*jmp(v))*dS    # SIP (interior)
a2 += eps*(-grad(u)*n*v - grad(v)*n*u + alpha/h*u*v)*dD                            # SIP Nitsche (Dirichlet)
a2 += -u*(wind*grad(v))*dx                                                         # convection (volume, IBP)
a2 += bn*IfPos(bn, u, u.Other())*jmp(v)*dS                                         # convection (upwind edge)
a2.Assemble()
f2 = LinearForm(eps*(-grad(v)*n*ud + alpha/h*ud*v)*dD).Assemble()                  # Dirichlet data -> rhs
uDG = GridFunction(V2)
uDG.vec.data = a2.mat.Inverse(V2.FreeDofs(), inverse="umfpack") * f2.vec
print(f"DG:  u in [{min(uDG.vec):.2f}, {max(uDG.vec):.2f}]   ({V2.ndof} dofs, all global)")
Draw(uDG, mesh, "u (DG)", min=0, max=1, autoscale=False)

The contrast is stark — $H^1$ rings, DG does not:

In [ ]:
gx = gy = np.linspace(-1, 1, 200)
def sample(gf):
    return np.array([[gf(mesh(float(xx), float(yy))) for xx in gx] for yy in gy])
fig, ax = plt.subplots(1, 2, figsize=(10, 4.6))
c0 = ax[0].contourf(gx, gy, np.clip(sample(uH1), -2, 2), levels=np.linspace(-2, 2, 21), cmap="RdBu_r")
ax[0].set_title("H1 — oscillating (clipped to ±2)"); fig.colorbar(c0, ax=ax[0])
c1 = ax[1].contourf(gx, gy, sample(uDG), levels=np.linspace(0, 1, 21), cmap="hot")
ax[1].set_title("upwind DG — clean"); fig.colorbar(c1, ax=ax[1])
for a in ax: a.set_aspect("equal"); a.set_xticks([]); a.set_yticks([])
plt.tight_layout()

## 3. HDG — a facet unknown, and static condensation

DG couples every element to its neighbours, so **all** its dofs are global. **Hybrid DG**
introduces a separate unknown $\hat u$ on the **facets** and lets elements talk **only** to
$\hat u$ (never directly to each other). The element-interior dofs then couple **only within
their element**, so they can be **eliminated locally** — *static condensation* (`condense=True`)
— leaving a much smaller global system in the **facet** unknowns alone.

In [ ]:
Vl = L2(mesh, order=order)
Vf = FacetFESpace(mesh, order=order, dirichlet="right|left")
X = Vl * Vf
(u, uhat), (v, vhat) = X.TnT()
dSb = dx(element_boundary=True)                         # integrate over each element's boundary
a3 = BilinearForm(X, condense=True)                    # <-- eliminate element interiors locally
a3 += eps*grad(u)*grad(v)*dx
a3 += eps*(-grad(u)*n*(v-vhat) - grad(v)*n*(u-uhat) + alpha/h*(u-uhat)*(v-vhat))*dSb   # HDG diffusion
a3 += -u*(wind*grad(v))*dx                                                             # convection (volume)
a3 += bn*IfPos(bn, u, uhat)*(v-vhat)*dSb                                               # upwind via facet
a3.Assemble()

uHDG = GridFunction(X)
uHDG.components[1].Set(ud, definedon=mesh.Boundaries("right|left"))
res = (-a3.mat * uHDG.vec).Evaluate()                  # condensed solve (Dirichlet kept):
res.data += a3.harmonic_extension_trans * res          #  1) lift element interiors to facets
uHDG.vec.data += a3.mat.Inverse(X.FreeDofs(coupling=True), inverse="umfpack") * res   # 2) global facet solve
uHDG.vec.data += a3.harmonic_extension * uHDG.vec      #  3) extend facets back into interiors
uHDG.vec.data += a3.inner_solve * res                  #  4) local interior solve
nglob = sum(X.FreeDofs(coupling=True))
print(f"HDG: u in [{min(uHDG.components[0].vec):.2f}, {max(uHDG.components[0].vec):.2f}]")
print(f"     global (condensed) dofs = {nglob}   vs DG's {Vl.ndof}  — the interiors are gone")
Draw(uHDG.components[0], mesh, "u (HDG)", min=0, max=1, autoscale=False)

Same clean solution as DG, but the **globally coupled** system is **smaller** — exactly why HDG
scales well for the 3-D flow problems in Part III.

**Next:** Part III puts these tools to work — first by **fusing** unsteady (unit 8) and
nonlinear (unit 9) into a pattern-forming reaction–diffusion system: **Turing**.

In [ ]:
# Navigation between units — shown only in a live notebook (Colab / JupyterLite /
# local Jupyter), never in the rendered website (which has its own prev/next nav).
import os, sys
if not os.environ.get("WEBGUI_SCENE_DIR"):          # not the static site build
    _prev = ("09-nonlinear-allencahn", "9 · Nonlinear problems — Allen–Cahn & Newton")
    _next = ("11-turing-patterns", "11 · How the beast got its stripes 🌈")
    def _u(_nb):
        if "google.colab" in sys.modules:
            return "https://colab.research.google.com/github/schruste/ngsum2026-colab/blob/colab/" + _nb + ".ipynb"
        return _nb + ".ipynb"                       # JupyterLite & local: relative .ipynb link
    _parts  = ["⬅️ **Previous:** [%s](%s)" % (_prev[1], _u(_prev[0]))] if _prev else []
    _parts += ["➡️ **Next:** [%s](%s)" % (_next[1], _u(_next[0]))] if _next else []
    from IPython.display import display, Markdown
    display(Markdown(" · ".join(_parts)))